# **Import & Parameters**

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql.types import StringType

spark = SparkSession.builder.getOrCreate()

BRONZE_LAKEHOUSE = "LH_Bronze"
SILVER_LAKEHOUSE = "LH_Silver"
SOURCE_FOLDER    = "sales"
CONFIG_TABLE     = "pipeline_config"

config_path = "Files/config_ingestion.csv"


from datetime import datetime
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, FloatType, TimestampType
)
import traceback

AUDIT_TABLE = "silver_audit_log"


# **ABFSS Path**

In [ ]:
bronze_info = notebookutils.lakehouse.get(BRONZE_LAKEHOUSE)
bronze_path = bronze_info["properties"]["abfsPath"]

silver_info = notebookutils.lakehouse.get(SILVER_LAKEHOUSE)
silver_path = silver_info["properties"]["abfsPath"]

print(f"📂 Bronze path : {bronze_path}")
print(f"📂 Silver path : {silver_path}")


# **Read & Filter configuration file with Sales rows**

In [ ]:
df_config = (
    spark.read
    .option("header",      "true")
    .option("inferSchema", "true")
    .option("sep",         ";")
    .csv(config_path)
)

rows = (
    df_config
    .filter(df_config.SourceFolder == SOURCE_FOLDER)
    .collect()
)

object_names = [row["DestinationTable"] for row in rows]
print(f"📋 Tabelle da processare: {len(object_names)} → {object_names}")


# **Define Cleaning Function**

In [ ]:

def pulisci_dataframe(df: DataFrame) -> DataFrame:

    espressioni = []

    for campo in df.schema.fields:
        nome = campo.name

        if isinstance(campo.dataType, StringType):
            # Trim + empty-string→NULL 
            col_pulita = F.trim(F.col(nome))
            col_pulita = F.when(col_pulita == "", None).otherwise(col_pulita)
            espressioni.append(col_pulita.alias(nome))
        else:
            # No changes for no text columns
            espressioni.append(F.col(nome))

    df_pulito = df.select(*espressioni)

    # After normalization proceed dropping rows with NULL or blank space
    df_pulito = df_pulito.dropna(how="all")

    return df_pulito


# **Write to Silver**

In [ ]:

print("=" * 65)
print("🚀 INIZIO PROCESSING BRONZE → SILVER")
print("=" * 65)

risultati = []   # Collect result of each table for the audit log

for row in rows:
    nome_tabella = row["DestinationTable"]
    nome_silver  = nome_tabella.replace("bronze_", "silver_")
    inizio       = datetime.now()   # ➕ NUOVO

    print(f"\n⏳ Processing: {nome_tabella} → {nome_silver}")

    try:
        # 1. Read Bronze via path ABFSS
        df_bronze = spark.read.format("delta").load(f"{bronze_path}/Tables/{nome_tabella}")
        righe_bronze = df_bronze.count()

        # 2. Apply cleaning function
        df_silver = pulisci_dataframe(df_bronze)
        righe_silver = df_silver.count()

        # 3. Full write in Silver Lakehouse
        (
            df_silver.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .save(f"{silver_path}/Tables/{nome_silver}")
        )

        fine     = datetime.now()                                 
        durata_s = round((fine - inizio).total_seconds(), 2)       

        print(f"   ✅ Completato in {durata_s}s")
        print(f"   ├── Righe Bronze : {righe_bronze:,}")
        print(f"   ├── Righe Silver : {righe_silver:,}")
        print(f"   └── Scartate     : {righe_bronze - righe_silver:,}")

        risultati.append({                                         
            "tabella"       : nome_silver,
            "righe_bronze"  : righe_bronze,
            "righe_silver"  : righe_silver,
            "righe_scartate": righe_bronze - righe_silver,
            "durata_sec"    : durata_s,
            "stato"         : "SUCCESS",
            "errore"        : "",       
            "processed_ts"  : fine
        })

    except Exception as e:
        fine = datetime.now()           
        msg  = traceback.format_exc()   
        print(f"   ❌ ERRORE su {nome_tabella}: {e}")

        risultati.append({               
            "tabella"       : nome_silver,
            "righe_bronze"  : 0,
            "righe_silver"  : 0,
            "righe_scartate": 0,
            "durata_sec"    : round((fine - inizio).total_seconds(), 2),
            "stato"         : "FAILED",
            "errore"        : msg,
            "processed_ts"  : fine
        })
        continue

print("\n" + "=" * 65)
print("🏁 PROCESSING COMPLETATO")
print("=" * 65)


# **Saving Audit Log**

In [ ]:

# Same schema as for the notebook used for ERP tables
schema_audit = StructType([
    StructField("tabella",        StringType(),    True),
    StructField("righe_bronze",   IntegerType(),   True),
    StructField("righe_silver",   IntegerType(),   True),
    StructField("righe_scartate", IntegerType(),   True),
    StructField("durata_sec",     FloatType(),     True),
    StructField("stato",          StringType(),    True),
    StructField("errore",         StringType(),    True),
    StructField("processed_ts",   TimestampType(), True),
])

successi = [r for r in risultati if r["stato"] == "SUCCESS"]
falliti  = [r for r in risultati if r["stato"] == "FAILED"]

print(f"\n📊 RIEPILOGO ESECUZIONE")
print(f"   ✅ Tabelle OK     : {len(successi)}/{len(risultati)}")
print(f"   ❌ Tabelle Fallite: {len(falliti)}/{len(risultati)}")

if falliti:
    print(f"\n⚠  TABELLE CON ERRORI:")
    for r in falliti:
        print(f"   → {r['tabella']}: {r['errore'][:120]}...")

df_audit = spark.createDataFrame(risultati, schema=schema_audit)

(
    df_audit.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .save(f"{silver_path}/Tables/{AUDIT_TABLE}")
)

print(f"\n📝 Audit log salvato in: Silver → {AUDIT_TABLE}")

if falliti:
    raise Exception(
        f"❌ {len(falliti)} tabelle non processate. Consulta '{AUDIT_TABLE}' per i dettagli."
    )
